In [1]:
import sys
sys.path.append("../")
# import config
import os
import numpy as np
import random

import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report
import time

# import DATA_CODE.DATA_LOADER as DATA_LOADER
from torch.utils.data.dataset import Dataset

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Define the paths and other parameters
experiment_id = 'train'
sat_data_dir_path = '/content/drive/MyDrive/CalCrop_Data/Satellite/'
wea_data_dir_path = '/content/drive/MyDrive/CalCrop_Data/Weather/'
label_data_dir_path = '/content/drive/MyDrive/CalCrop_Data/Label_Eroded/'
distri_dir_path = ''
model_dir = '/content/drive/MyDrive/CalCrop_Data/Model'
in_channels = 10
in_channels_w = 7
out_channels = 33
unknown_class = 100
learning_rate = 0.0001
n_epochs = 10


input_patch_size = 32
output_patch_size = 32
batch_size = 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


cuda


In [3]:
import torch

class WSTATT(torch.nn.Module):
    def __init__(self, in_channels, in_channels_w, out_channels):
        super(WSTATT, self).__init__()  # Initialize parent class (torch.nn.Module)

        # Feature dimensions for spatial and weather data processing
        self.s_fe = 64  # Spatial feature base dimension
        self.w_fe = 8   # Weather feature base dimension
        s_fe = self.s_fe
        w_fe = self.w_fe

        # Spatial Encoder Path (Downsampling)
        # Each block: Conv -> ReLU -> Conv -> ReLU -> MaxPool
        self.conv1_1 = torch.nn.Conv2d(in_channels, s_fe, 3, padding=1)  # First convolution
        self.conv1_2 = torch.nn.Conv2d(s_fe, s_fe, 3, padding=1)         # Second convolution
        self.conv2_1 = torch.nn.Conv2d(s_fe, s_fe*2, 3, padding=1)       # Increase features
        self.conv2_2 = torch.nn.Conv2d(s_fe*2, s_fe*2, 3, padding=1)     # Maintain features
        self.conv3_1 = torch.nn.Conv2d(s_fe*2, s_fe*4, 3, padding=1)     # Double features again
        self.conv3_2 = torch.nn.Conv2d(s_fe*4, s_fe*4, 3, padding=1)     # Maintain features

        # Decoder Path (Upsampling) with Skip Connections
        self.unpool2_cat = torch.nn.ConvTranspose2d(s_fe*8 + w_fe*8, s_fe*2, kernel_size=2, stride=2)  # Upsample + feature reduction
        self.upconv2_1 = torch.nn.Conv2d(s_fe*4, s_fe*2, 3, padding=1)   # Reduce features after skip connection
        self.upconv2_2 = torch.nn.Conv2d(s_fe*2, s_fe*2, 3, padding=1)   # Process features
        self.unpool1 = torch.nn.ConvTranspose2d(s_fe*2, s_fe, kernel_size=2, stride=2)  # Final upsampling
        self.upconv1_1 = torch.nn.Conv2d(s_fe*2, s_fe, 3, padding=1)     # Reduce features
        self.upconv1_2 = torch.nn.Conv2d(s_fe, s_fe, 3, padding=1)       # Final processing

        # Output layer (1x1 convolution for channel reduction)
        self.out = torch.nn.Conv2d(s_fe, out_channels, kernel_size=1, padding=0)

        # Shared Operations
        self.maxpool = torch.nn.MaxPool2d(2)    # Downsamples by 2x
        self.relu = torch.nn.ReLU()             # Activation function
        self.dropout = torch.nn.Dropout(p=0.1)  # Regularization

        # Temporal Processing (LSTMs)
        # For spatial features: bidirectional LSTM
        self.lstm = torch.nn.LSTM(s_fe*4, s_fe*4, batch_first=True, bidirectional=True)
        # For weather data: bidirectional LSTM
        self.lstm_w = torch.nn.LSTM(in_channels_w, w_fe*4, batch_first=True, bidirectional=True)

        # Attention Mechanism
        self.attention_encode = torch.nn.Linear(s_fe*8 + w_fe*8, 1)  # Computes attention scores

        # Weight Initialization
        for m in self.modules():
            if isinstance(m, torch.nn.Conv2d) or isinstance(m, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(m.weight)  # Xavier init for better convergence

    def crop_and_concat(self, x1, x2):
        """Aligns and concatenates encoder features (x1) with decoder features (x2)"""
        # Calculate cropping offsets for center alignment
        offset_2 = (x1.shape[2] - x2.shape[2]) // 2  # Height offset
        offset_3 = (x1.shape[3] - x2.shape[3]) // 2  # Width offset

        # Crop the larger tensor (x1) to match x2's spatial dimensions
        x1_crop = x1[:, :, offset_2:offset_2+x2.shape[2], offset_3:offset_3+x2.shape[3]]

        # Concatenate along channel dimension
        return torch.cat([x1_crop, x2], dim=1)

    def forward(self, x_s, x_w):
        """Forward pass with spatial data (x_s) and weather data (x_w)"""
        s_fe = self.s_fe
        w_fe = self.w_fe

        # Reshape inputs: Combine batch and sequence dimensions
        batch_size_s, seq_len_s, channels_s, input_patch_size, _ = x_s.shape
        batch_size_w, seq_len_w, channels_w, _, _ = x_w.shape
        x_s = x_s.view(-1, channels_s, input_patch_size, input_patch_size)  # [batch*seq, C, H, W]
        x_w = x_w.view(-1, seq_len_w, channels_w)  # [batch, seq_w, C_w] (after view)

        # --- Spatial Encoder Path ---
        # Block 1
        conv1 = self.relu(self.conv1_1(x_s))
        conv1 = self.relu(self.conv1_2(conv1))
        maxpool1 = self.maxpool(conv1)  # Downsample 2x

        # Block 2
        conv2 = self.relu(self.conv2_1(maxpool1))
        conv2 = self.relu(self.conv2_2(conv2))
        maxpool2 = self.maxpool(conv2)  # Downsample 4x from original

        # Block 3 (Deepest features)
        conv3 = self.relu(self.conv3_1(maxpool2))
        conv3 = self.relu(self.conv3_2(conv3))  # Now at 1/4 original resolution

        # --- Temporal Processing ---
        # Prepare spatial features for LSTM: [batch*H*W, seq_len, features]
        shape_enc = conv3.shape  # Save original shape for later reshaping
        conv3 = conv3.view(-1, seq_len_s, conv3.shape[1], conv3.shape[2]*conv3.shape[3])
        conv3 = conv3.permute(0, 3, 1, 2)  # [batch, H*W, seq_len, features]
        conv3 = conv3.reshape(conv3.shape[0]*conv3.shape[1], seq_len_s, s_fe*4)  # [batch*H*W, seq_len, features]

        # Process spatial sequence with LSTM
        lstm, _ = self.lstm(conv3)  # Output: [batch*H*W, seq_len, s_fe*8] (bidirectional)

        # Process weather data with LSTM
        lstm_w, _ = self.lstm_w(x_w)  # Output: [batch, seq_len_w, w_fe*8]

        # Align weather features with spatial sequence length
        lstm_subset = lstm_w[:, 15::15, :]  # Sample every 15th timestep
        lstm_subset = lstm_subset[:, :seq_len_s, :]  # Match spatial sequence length

        # Expand weather features to spatial dimensions
        lstm_subset = lstm_subset.reshape(lstm_subset.shape[0], lstm_subset.shape[1], lstm_subset.shape[2], 1, 1)
        lstm_subset = lstm_subset.repeat(1, 1, 1, int(input_patch_size/4), int(input_patch_size/4))  # Tile to H/4 x W/4
        lstm_subset = lstm_subset.view(-1, seq_len_s, lstm_subset.shape[2], lstm_subset.shape[3]*lstm_subset.shape[4])
        lstm_subset = lstm_subset.permute(0, 3, 1, 2)
        lstm_subset = lstm_subset.reshape(lstm_subset.shape[0]*lstm_subset.shape[1], seq_len_s, w_fe*8)  # [batch*H*W, seq_len, w_fe*8]

        # --- Attention Mechanism ---
        # Combine spatial and weather features
        lstm_concat = torch.cat([lstm, lstm_subset], dim=2)  # [batch*H*W, seq_len, s_fe*8 + w_fe*8]
        lstm_concat_relu = self.relu(lstm_concat.reshape(-1, w_fe*8 + s_fe*8))  # Flatten for linear layer

        # Compute attention weights
        attn_scores = self.attention_encode(torch.tanh(lstm_concat_relu))  # [batch*H*W*seq_len, 1]
        attn_scores = attn_scores.view(-1, shape_enc[2], shape_enc[3], seq_len_s)  # [batch, H, W, seq_len]
        attn_scores = attn_scores.permute(0, 3, 1, 2)  # [batch, seq_len, H, W]
        attention_weights = torch.nn.functional.softmax(
            torch.squeeze(torch.nn.functional.avg_pool2d(attn_scores, shape_enc[2])),  # Average over H/W
            dim=1  # Softmax over sequence dimension
        )  # [batch, seq_len]

        # Create context vector (weighted sum of features)
        attn_weights_4d = attention_weights.view(-1, 1, 1, seq_len_s).repeat(1, shape_enc[2], shape_enc[3], 1)
        weighted_features = (attn_weights_4d.view(-1, 1) * lstm_concat_relu)  # Element-wise multiplication
        context = torch.sum(weighted_features.view(-1, seq_len_s, s_fe*8 + w_fe*8), dim=1)  # Sum over sequence
        context = context.view(-1, shape_enc[2], shape_enc[3], s_fe*8 + w_fe*8).permute(0, 3, 1, 2)  # [batch, C, H, W]

        # Detach attention weights to stop gradient flow during decoder aggregation
        attention_weights_fixed = attention_weights.detach()

        # --- Decoder Path ---
        # Upsample context features
        unpool2_cat = self.unpool2_cat(context)  # Upsample 2x to 1/2 original resolution

        # Aggregate encoder features using attention weights
        conv2_view = conv2.view(-1, seq_len_s, conv2.shape[1], conv2.shape[2], conv2.shape[3])
        agg_conv2 = torch.sum(attention_weights_fixed.view(-1, seq_len_s, 1, 1, 1) * conv2_view, dim=1)

        # Combine with upsampled context
        concat2 = self.crop_and_concat(agg_conv2, unpool2_cat)  # Skip connection
        upconv2 = self.relu(self.upconv2_1(concat2))
        upconv2 = self.relu(self.upconv2_2(upconv2))

        # Final upsampling block
        unpool1 = self.unpool1(upconv2)  # Upsample to original resolution

        # Aggregate first encoder block features
        conv1_view = conv1.view(-1, seq_len_s, conv1.shape[1], conv1.shape[2], conv1.shape[3])
        agg_conv1 = torch.sum(attention_weights_fixed.view(-1, seq_len_s, 1, 1, 1) * conv1_view, dim=1)

        # Combine features
        concat1 = self.crop_and_concat(agg_conv1, unpool1)  # Skip connection
        upconv1 = self.relu(self.upconv1_1(concat1))
        upconv1 = self.relu(self.upconv1_2(upconv1))

        # Output layer
        out = self.out(upconv1)  # Final 1x1 convolution
        return out

In [4]:
import numpy as np
from torch.utils.data import Dataset

def create_patches_satellite_weather_labels(grid):
    """Create image patches, weather data, and label patches for a given grid location.

    Args:
        grid: Identifier for the geographical area to process

    Returns:
        Tuple of (image_patches, weather_patches, label_patches) as numpy arrays
    """
    # Start timing for performance measurement
    time_start = time.time()

    # Load satellite image data (4D array: [timesteps, channels, height, width])
    image = np.load(os.path.join(sat_data_dir_path, grid + "_image.npy"))

    # Load label data (2D array: [height, width] - ground truth for each pixel)
    label = np.load(os.path.join(label_data_dir_path, grid + "_label.npy"))

    # Load weather data (3D array: [timesteps, weather_features])
    weather = np.load(os.path.join(wea_data_dir_path, grid + '_weather.npy'))

    # Handle missing values in weather data by replacing NaNs with 0
    weather[np.isnan(weather)] = 0

    # Get dimensions of the label data (same as full image size)
    height, width = label.shape

    # Calculate padding difference needed between input and output patches
    # This ensures the input patch is larger than the output label patch
    diff = (input_patch_size - output_patch_size) // 2

    # Initialize lists to store patches
    image_patches = []   # Will store satellite image patches
    weather_patches = []  # Will store weather data sequences
    label_patches = []    # Will store label patches

    # Slide a window over the image to create patches
    # Step size is output_patch_size (non-overlapping output patches)
    for i in range(height // output_patch_size):        # Vertical steps
        for j in range(width // output_patch_size):     # Horizontal steps
            # Calculate label patch boundaries (central region)
            i_label_start = i * output_patch_size
            i_label_end = (i + 1) * output_patch_size
            j_label_start = j * output_patch_size
            j_label_end = (j + 1) * output_patch_size

            # Calculate larger image patch boundaries (with padding)
            i_image_start = i_label_start - diff
            i_image_end = i_label_end + diff
            j_image_start = j_label_start - diff
            j_image_end = j_label_end + diff

            # Check if the image patch is within the original image boundaries
            if (0 <= i_image_start < height and
                0 <= i_image_end <= height and
                0 <= j_image_start < width and
                0 <= j_image_end <= width):

                # Extract satellite image patch (all timesteps and channels)
                # Shape: [timesteps, channels, patch_height, patch_width]
                image_patch = image[:, :, i_image_start:i_image_end, j_image_start:j_image_end]

                # Extract corresponding label patch (ground truth)
                # Shape: [patch_height, patch_width]
                label_patch = label[i_label_start:i_label_end, j_label_start:j_label_end]

                # Store extracted patches
                image_patches.append(image_patch)
                label_patches.append(label_patch)

                # Associate the entire weather sequence with this spatial patch
                # Same weather data used for all patches from this grid
                weather_patches.append(weather)

    # Convert lists to numpy arrays for efficient processing
    image_patches = np.array(image_patches).astype(np.float32)     # Shape: [num_patches, timesteps, channels, H, W]
    weather_patches = np.array(weather_patches).astype(np.float32) # Shape: [num_patches, timesteps, weather_features]
    label_patches = np.array(label_patches).astype(np.int8)        # Shape: [num_patches, H, W]

    return image_patches, weather_patches, label_patches


class SEGMENTATION_WEATHER(Dataset):
    """Custom PyTorch Dataset for satellite image patches with weather data.

    Combines:
    - Satellite image patches (spatio-temporal data)
    - Weather data (temporal features)
    - Segmentation labels (ground truth)
    """

    def __init__(self, image_patches, weather_patches, label_patches):
        """Initialize dataset with preprocessed patches.

        Args:
            image_patches: Array of satellite image patches
                shape: (num_samples, timesteps, channels, height, width)
            weather_patches: Array of weather data sequences
                shape: (num_samples, timesteps, weather_features)
            label_patches: Array of segmentation labels
                shape: (num_samples, height, width)
        """
        self.image_patches = image_patches
        self.weather_patches = weather_patches
        self.label_patches = label_patches

    def __len__(self):
        """Return total number of samples in dataset."""
        return len(self.label_patches)  # Number of patches determines dataset size

    def __getitem__(self, index):
        """Retrieve a single sample by index.

        Returns:
            tuple: (satellite_patch, weather_data, label_patch)
        """
        # Return data for a specific patch location
        return (
            self.image_patches[index],   # Satellite data: [timesteps, channels, H, W]
            self.weather_patches[index], # Weather data: [timesteps, features]
            self.label_patches[index]    # Label: [H, W] (ground truth segmentation)
        )

In [5]:
print("#######################################################################")
print("BUILD MODEL")
# Initialize the WSTATT model with specified input/output channels
model = WSTATT(
    in_channels=in_channels,        # Number of input channels in satellite images
    in_channels_w=in_channels_w,    # Number of features in weather data
    out_channels=out_channels       # Number of output classes for segmentation
)
model = model.to(device)  # Move model to GPU if available


# Define loss function (CrossEntropy) and optimizer (Adam)
criterion = torch.nn.CrossEntropyLoss(ignore_index=unknown_class)  # Ignore unlabeled pixels
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # Adam optimizer


print("#######################################################################")
print("TRAIN MODEL")
train_loss = []  # Track loss per epoch


# Main training loop
for epoch in range(1, n_epochs + 1):
    print('## EPOCH {} ##'.format(epoch))

    model.train()  # Set model to training mode
    train_time_start = time.time()  # Start epoch timer

    # Normally you'd have multiple grids, but here we're using one for demonstration
    dataset = ['T11SKA_2018_1_2']  # Single grid identifier
    dataset = random.sample(dataset, len(dataset))  # Shuffle (trivial with one item)

    epoch_loss = 0  # Accumulate loss across all grids
    # Process each geographic grid
    for grid_num, grid in enumerate(dataset):
        grid_time_start = time.time()  # Start grid timer

        # Create patches for current grid
        image_patches, weather_patches, label_patches = create_patches_satellite_weather_labels(grid)

        # Create PyTorch Dataset
        data = SEGMENTATION_WEATHER(image_patches, weather_patches, label_patches)

        # Create DataLoader for batching
        data_loader = torch.utils.data.DataLoader(
            dataset=data,
            batch_size=batch_size,
            shuffle=True,       # Shuffle patches within grid
            num_workers=0,      # No parallel loading (0 for debugging)
            drop_last=False      # Keep last incomplete batch
        )

        grid_loss = 0  # Accumulate loss for this grid
        # Process batches
        for batch, [image_patch, weather_patch, label_patch] in enumerate(data_loader):
            # Reset gradients
            optimizer.zero_grad()

            # Forward pass - model prediction
            # Note: image_patch shape: [batch, timesteps, channels, H, W]
            #        weather_patch shape: [batch, timesteps, features]
            out = model(image_patch.to(device), weather_patch.to(device))

            # Prepare labels: convert to long tensor and move to device
            label_patch = label_patch.type(torch.long).to(device)

            # Calculate loss between prediction and ground truth
            batch_loss = criterion(out, label_patch)

            # Backpropagation
            batch_loss.backward()       # Calculate gradients
            optimizer.step()            # Update weights

            # Accumulate batch loss
            grid_loss += batch_loss.item()

        # Calculate average loss per batch for this grid
        grid_loss = grid_loss / (batch + 1)  # +1 because batch starts at 0
        print('\tGrid No:{}\tGrid:{}\tGrid Loss:{:.4f}\tGrid Time:{:.2f}s'.format(
            grid_num, grid, grid_loss, time.time() - grid_time_start
        ))
        epoch_loss += grid_loss

    # Calculate average loss across all grids
    epoch_loss = epoch_loss / (grid_num + 1)
    print('\nTrain Loss:{:.4f}  Train Time:{:.2f}s'.format(
        epoch_loss, time.time() - train_time_start
    ), end="\n")
    train_loss.append(epoch_loss)  # Store for later analysis

    if(epoch_loss < train_loss[-1] or len(train_loss)==1): #This is to make sure the model is not overfitting. You usually do this with a validation set and not the train set
      torch.save(model.state_dict(), "Model.pt") #You also might want to consider saving  every epochs




#######################################################################
BUILD MODEL
#######################################################################
TRAIN MODEL
## EPOCH 1 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:1.9378	Grid Time:13.30s

Train Loss:1.9378  Train Time:13.31s
## EPOCH 2 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:1.0274	Grid Time:14.74s

Train Loss:1.0274  Train Time:14.74s
## EPOCH 3 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.8519	Grid Time:13.88s

Train Loss:0.8519  Train Time:13.88s
## EPOCH 4 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.7789	Grid Time:14.54s

Train Loss:0.7789  Train Time:14.55s
## EPOCH 5 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.7446	Grid Time:13.96s

Train Loss:0.7446  Train Time:13.96s
## EPOCH 6 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.7136	Grid Time:13.25s

Train Loss:0.7136  Train Time:13.25s
## EPOCH 7 ##
	Grid No:0	Grid:T11SKA_2018_1_2	Grid Loss:0.7171	Grid Time:12.98s

Train Loss:0.7171  Train Time:12.98s
## EPO

In [6]:
threshold = 50000
labels_list = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32]
class_names = ['Corn','Cotton','Rice','Sunflower','Barley','Winter_Wheat','Safflower','Dry Beans','Onions','Tomatoes','Cherries','Grapes','Citrus','Almonds','Walnut','Pistachio','Garlic','Olives','Pomegranates','Alfalfa','Hay','Barren_land','Fallow_and_Idle','Deciduous_Forests','Evergreen_forest','Mixed_Forests','Clover_and_wildflower','Shrubland','Grass','Woody_wetlands','Herbaceous_Wetlands','Water','Urban']

In [7]:
print("#######################################################################")
print("TEST MODEL")
model = WSTATT(in_channels=in_channels, in_channels_w=in_channels_w, out_channels=out_channels)
model = model.to(device)

print("LOAD MODEL")
model.load_state_dict(torch.load("Model.pt"),strict = False) #This loads the model we saved
# Define loss function (same as training, ignoring unlabeled pixels)
criterion = torch.nn.CrossEntropyLoss(ignore_index=unknown_class)

print("#######################################################################")

# Initialize metrics storage
test_loss = []    # Track loss per test run
label_list = []   # Collect all ground truth labels
pred_list = []    # Collect all model predictions

# Set model to evaluation mode (disables dropout/BatchNorm)
model.eval()
test_time_start = time.time()  # Start test timer

# Test dataset - normally multiple grids, here using one for demonstration
dataset = ['T11SKA_2019_2_2']  # Grid identifier

epoch_loss = 0  # Accumulate loss across grids
# Process each grid in test dataset
for grid_num, grid in enumerate(dataset):
    # Create patches for current grid
    image_patches, weather_patches, label_patches = create_patches_satellite_weather_labels(grid)

    # Create PyTorch Dataset
    data = SEGMENTATION_WEATHER(image_patches, weather_patches, label_patches)

    # Create DataLoader for batching
    data_loader = torch.utils.data.DataLoader(
        dataset=data,
        batch_size=batch_size,
        shuffle=True,       # Shuffle patches (optional for testing)
        num_workers=0,      # No parallel loading
        drop_last=False      # Keep the last incomplete batch
    )

    grid_loss = 0  # Accumulate loss for this grid
    # Process all batches in grid
    for batch, [image_patch, weather_patch, label_patch] in enumerate(data_loader):
        # Forward pass WITHOUT gradient calculation (saves memory)
        with torch.no_grad():  # Disable gradient tracking
            patch_out = model(image_patch.to(device), weather_patch.to(device))

        # Convert model outputs to probabilities using softmax
        # dim=1 applies softmax across classes (channels dimension)
        patch_prob_out = torch.nn.functional.softmax(patch_out, dim=1)

        # Convert to numpy for processing (detach from computation graph)
        patch_prob_out_numpy = patch_prob_out.cpu().detach().numpy()

        # Get predicted class (index with highest probability)
        pred_patch = np.argmax(patch_prob_out_numpy, axis=1)  # Shape: [batch, H, W]

        # Prepare labels for loss calculation
        label_patch_device = label_patch.type(torch.long).to(device)

        # Calculate loss
        batch_loss = criterion(patch_out, label_patch_device)
        grid_loss += batch_loss.item()  # Accumulate batch loss

        # Flatten predictions and labels to 1D arrays
        pred_patch_flat = np.reshape(pred_patch, (-1))        # [batch*H*W]
        label_patch_flat = np.reshape(label_patch, (-1))      # [batch*H*W]

        # Filter out unknown_class pixels (ignore_index)
        valid_mask = label_patch_flat != unknown_class
        pred_grid_flat = pred_patch_flat[valid_mask]
        label_grid_flat = label_patch_flat[valid_mask]

        # Collect valid predictions and labels for overall metrics
        for l in range(pred_grid_flat.shape[0]):
            label_list.append(label_grid_flat[l])
            pred_list.append(pred_grid_flat[l])

    # Calculate average loss for current grid
    grid_loss = grid_loss / (batch + 1)
    print('\tGrid No:{}\tGrid:{}\tGrid Loss:{:.4f}'.format(
        grid_num, grid, grid_loss
    ))
    epoch_loss += grid_loss

# Convert collected results to numpy arrays
label_array = np.array(label_list)  # All ground truth labels
pred_array = np.array(pred_list)    # All model predictions

# Calculate overall test loss
epoch_loss = epoch_loss / (grid_num + 1)
print('Test Loss:{:.4f}  Test Time:{:.2f}s'.format(
    epoch_loss, time.time() - test_time_start
))
test_loss.append(epoch_loss)  # Store for later analysis


# print('Overall unknown:', np.sum(pred_array == 100), '  labels:', np.sum(label_array == 100))
# print(classification_report(label_array, pred_array, target_names=class_names, digits=4,labels = labels_list))


# Compute support (i.e., the number of occurrences per class in label_array)
unique_labels, support = np.unique(label_array, return_counts=True)

# Filter labels with support above 50,000
valid_labels = unique_labels[support > threshold]

# Create a filtered class name list
filtered_class_names = [class_names[i] for i in range(len(class_names)) if labels_list[i] in valid_labels]

# Compute classification report only for selected labels
print(classification_report(label_array, pred_array, target_names=filtered_class_names, digits=4, labels=valid_labels))


#######################################################################
TEST MODEL
LOAD MODEL
#######################################################################
	Grid No:0	Grid:T11SKA_2019_2_2	Grid Loss:1.4612
Test Loss:1.4612  Test Time:22.54s
              precision    recall  f1-score   support

      Grapes     0.5994    0.9999    0.7495    551327
     Almonds     0.6320    0.0010    0.0019    221218
       Urban     0.0000    0.0000    0.0000     94433

   micro avg     0.5995    0.6361    0.6172    866978
   macro avg     0.4105    0.3336    0.2505    866978
weighted avg     0.5425    0.6361    0.4771    866978



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
